# Airline Passenger Satisfaction: Classification

Goal: predict whether a passenger was **satisfied** or **neutral/dissatisfied** after a flight, using their profile, flight details and in-flight service ratings. The notebook compares four models on the same train/test split, tunes the best tree-based one, and looks at which features drive satisfaction.

Dataset: Airline Passenger Satisfaction (103,904 rows, 25 columns).

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_style("whitegrid")
RANDOM_STATE = 42

# Hyperparameter search takes a few minutes on the full data. Set to False to skip it.
RUN_TUNING = True

## 2. Load the data

In [ ]:
# Works in Colab (Drive) and locally (CSV sitting next to the notebook).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_PATH = "/content/drive/MyDrive/kaushik/Airline data set.csv"
except ImportError:
    DATA_PATH = "Airline data set.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

## 3. Data quality checks

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

missing = pd.DataFrame({
    "missing": df.isnull().sum(),
    "percent": (df.isnull().mean() * 100).round(2),
})
print(missing[missing["missing"] > 0])

Only `Arrival Delay in Minutes` has missing values. I am **not** filling them here. The imputer sits inside the modelling pipeline instead, so the fill value (median) is learned from the training rows only and nothing from the test set leaks in. Median is used because delays are heavily right-skewed.

`Unnamed: 0` is just a row counter and `id` is a passenger identifier. Neither describes the passenger or the flight, so both are dropped.

In [ ]:
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
df = df.drop_duplicates()
print("Shape after dropping identifier columns:", df.shape)

## 4. Target and quick EDA

In [ ]:
from pathlib import Path

DATA_PATH = Path("Airline data set.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_PATH = Path("/content/drive/MyDrive/kaushik/Airline data set.csv")
    except ImportError:
        pass

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. Place 'Airline data set.csv' next to this notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=df, x="satisfaction", ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].set_xlabel("")

sns.barplot(data=df, x="Class", y="satisfied", ax=axes[1], errorbar=None)
axes[1].set_title("Share satisfied by travel class")
axes[1].set_ylabel("Share satisfied")

sns.barplot(data=df, x="Type of Travel", y="satisfied", ax=axes[2], errorbar=None)
axes[2].set_title("Share satisfied by type of travel")
axes[2].set_ylabel("Share satisfied")

plt.tight_layout()
plt.show()

In [ ]:
# Delay columns: how correlated are they?
print(df[["Departure Delay in Minutes", "Arrival Delay in Minutes"]].corr().round(3))

Departure and arrival delay move almost together, so a combined "total delay" feature would add very little and would only split the importance between three correlated columns. I keep the two original columns and skip extra engineered features. Tree models also do not need the age or distance bins used earlier.

## 5. Train/test split and preprocessing

In [ ]:
X = df.drop(columns=["satisfaction", "satisfied"])
y = df["satisfied"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Categorical:", cat_cols)
print("Numeric count:", len(num_cols))

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

## 6. Model comparison

All four models go through the same pipeline (preprocessing + model) and the same split. For each one I report train accuracy, 5-fold cross-validated accuracy on the training set, and test-set metrics. A large gap between train and test accuracy is a sign of overfitting.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(dual=False, max_iter=5000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                            n_jobs=-1, random_state=RANDOM_STATE),
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate(name, estimator):
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", estimator)])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    if hasattr(pipe, "predict_proba"):
        score = pipe.predict_proba(X_test)[:, 1]
    else:
        score = pipe.decision_function(X_test)

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv5, scoring="accuracy")

    row = {
        "Model": name,
        "Train Acc": accuracy_score(y_train, pipe.predict(X_train)),
        "CV Acc (mean)": cv_scores.mean(),
        "CV Acc (std)": cv_scores.std(),
        "Test Acc": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, score),
    }
    return pipe, row

fitted, rows = {}, []
for name, est in models.items():
    fitted[name], row = evaluate(name, est)
    rows.append(row)
    print(f"done: {name}")

results = pd.DataFrame(rows).set_index("Model").sort_values("Test Acc", ascending=False)
results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = results.reset_index()
sns.barplot(data=plot_df, x="Model", y="Test Acc", hue="Model", palette="Blues_d", legend=False, ax=ax)
for i, v in enumerate(plot_df["Test Acc"]):
    ax.text(i, v + 0.003, f"{v:.3f}", ha="center")
ax.set_ylim(max(0, plot_df["Test Acc"].min() - 0.05), 1.0)
ax.set_title("Test accuracy by model (same split, same preprocessing)")
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 7. Tuning the Random Forest

Randomized search with 3-fold CV on the **training set only**. The test set is not touched until the final evaluation. If tuning is switched off, the baseline Random Forest from section 6 is used instead.

In [ ]:
baseline_rf = fitted["Random Forest"]

if RUN_TUNING:
    rf_pipe = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                         n_jobs=-1, random_state=RANDOM_STATE)),
    ])
    param_dist = {
        "model__max_depth": [None, 15, 25],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.5],
    }
    search = RandomizedSearchCV(rf_pipe, param_dist, n_iter=6, cv=3, scoring="accuracy",
                                random_state=RANDOM_STATE, n_jobs=1, refit=True)
    search.fit(X_train, y_train)
    print("Best params:", search.best_params_)
    print("Best CV accuracy:", round(search.best_score_, 4))
    final_model = search.best_estimator_
else:
    final_model = baseline_rf

## 8. Final model evaluation (test set)

In [ ]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print("Test accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Baseline RF   :", round(accuracy_score(y_test, baseline_rf.predict(X_test)), 4))
print("ROC-AUC       :", round(roc_auc_score(y_test, y_prob), 4))
print()
print(classification_report(y_test, y_pred, target_names=["Neutral/Dissatisfied", "Satisfied"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=["Neutral/Dissatisfied", "Satisfied"],
            yticklabels=["Neutral/Dissatisfied", "Satisfied"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion matrix")

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 9. Feature importance

Two views. The first is the Random Forest's built-in importance on the encoded features (so `Class` shows up as separate `Class_Business`, `Class_Eco` columns). The second is permutation importance on the test set, grouped by the original column. It measures how much accuracy drops when a column is shuffled, and it is less biased toward numeric or high-cardinality features.

In [ ]:
prep = final_model.named_steps["prep"]
rf_model = final_model.named_steps["model"]

names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
feat_imp = pd.Series(rf_model.feature_importances_, index=names).sort_values(ascending=False)

top15 = feat_imp.head(15)
plt.figure(figsize=(8, 6))
sns.barplot(x=top15.values, y=top15.index, color="steelblue")
plt.title("Top 15 features (Random Forest importance, encoded)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

top15.round(4)

In [ ]:
# Permutation importance on a 5,000-row sample of the test set (keeps runtime reasonable)
sample_idx = X_test.sample(n=min(5000, len(X_test)), random_state=RANDOM_STATE).index
perm = permutation_importance(final_model, X_test.loc[sample_idx], y_test.loc[sample_idx],
                              n_repeats=5, random_state=RANDOM_STATE, scoring="accuracy", n_jobs=1)

perm_imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
top_perm = perm_imp.head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=top_perm.values, y=top_perm.index, color="darkseagreen")
plt.title("Top 15 features (permutation importance, original columns)")
plt.xlabel("Mean drop in accuracy when shuffled")
plt.tight_layout()
plt.show()

top_perm.round(4)

## 10. Predicting a new passenger

In [ ]:
# Raw columns only. The pipeline handles imputation, scaling and encoding.
new_passenger = {
    "Gender": "Male",
    "Customer Type": "Loyal Customer",
    "Age": 30,
    "Type of Travel": "Business travel",
    "Class": "Business",
    "Flight Distance": 1200,
    "Inflight wifi service": 5,
    "Departure/Arrival time convenient": 4,
    "Ease of Online booking": 4,
    "Gate location": 3,
    "Food and drink": 4,
    "Online boarding": 5,
    "Seat comfort": 4,
    "Inflight entertainment": 5,
    "On-board service": 5,
    "Leg room service": 4,
    "Baggage handling": 5,
    "Checkin service": 5,
    "Inflight service": 5,
    "Cleanliness": 5,
    "Departure Delay in Minutes": 0,
    "Arrival Delay in Minutes": 0,
}

new_df = pd.DataFrame([new_passenger])[X_train.columns]   # same column order as training
prob = final_model.predict_proba(new_df)[0, 1]
label = "Satisfied" if prob >= 0.5 else "Neutral/Dissatisfied"
print(f"Prediction: {label}  (P(satisfied) = {prob:.2f})")

## 11. Summary

In [ ]:
best_baseline = results.index[0]
print(f"Best baseline model : {best_baseline} ({results.loc[best_baseline, 'Test Acc']:.4f} test accuracy)")
print(f"Final model (RF)    : {accuracy_score(y_test, y_pred):.4f} test accuracy, "
      f"{roc_auc_score(y_test, y_prob):.4f} ROC-AUC")
print()
print("Top 5 features by permutation importance:")
for feat, val in perm_imp.head(5).items():
    print(f"  {feat}: {val:.4f}")

**Notes and limits**

- The service ratings are collected from the passenger after the flight, so this model explains satisfaction rather than predicting it before departure.
- A rating of 0 in this dataset means "not applicable", but it is treated here as an ordinary numeric value.
- Results come from a single train/test split. The cross-validation columns in section 6 give a sense of how stable they are.
- Feature importance shows association, not cause. Improving wifi or boarding will not necessarily move satisfaction by the amount the chart suggests.